# Neural Machine Translation Transformer (NMT Transformer) Model: Version 2
## Myanmar (Burmese) Language: Grapheme to Phoneme Translation

## Setup

In [1]:
!marian --version

v1.12.0 65bf82ff 2023-02-21 09:56:29 -0800


In [2]:
from pathlib import Path
from IPython.display import Image

In [3]:
ROOT = Path("..").resolve()

DATA_ROOT = ROOT / "data"
DATA_DIR = DATA_ROOT / "cleaned"
VOCAB_DIR = DATA_ROOT / "vocab"

MODEL_ROOT = ROOT / "models"
MODEL_1_DIR = MODEL_ROOT / "baseline.transformer.myph"
MODEL_2_DIR = MODEL_ROOT / "change1.transformer.myph"
RESULT_DIR = MODEL_ROOT / "results"

BLEU_DIR = "/home/lawun330/NLP/ubuntu-17.04/moses/scripts/generic"

## Analysis: Model 1

In [4]:
!cat {MODEL_1_DIR}/valid.log

[2026-05-29 14:52:43] [valid] Ep. 80 : Up. 5000 : cross-entropy : 1.86253 : new best
[2026-05-29 14:52:44] [valid] Ep. 80 : Up. 5000 : perplexity : 1.62341 : new best
[2026-05-29 14:52:44] [valid] First sentence's tokens as scored:
[2026-05-29 14:52:44] [valid] DefaultVocab keeps original segments for scoring
[2026-05-29 14:52:44] [valid]   Hyp: ou' ta- ra. hpa- la- gu. ni
[2026-05-29 14:52:44] [valid]   Ref: ou' ta- ra. hpa- la. gu. ni
[2026-05-29 14:52:48] [valid] Ep. 80 : Up. 5000 : bleu : 72.5211 : new best
[2026-05-29 15:24:56] [valid] Ep. 160 : Up. 10000 : cross-entropy : 2.13194 : stalled 1 times (last best: 1.86253)
[2026-05-29 15:24:57] [valid] Ep. 160 : Up. 10000 : perplexity : 1.74127 : stalled 1 times (last best: 1.62341)
[2026-05-29 15:25:01] [valid] Ep. 160 : Up. 10000 : bleu : 74.4712 : new best
[2026-05-29 15:52:43] [valid] Ep. 240 : Up. 15000 : cross-entropy : 2.28535 : stalled 2 times (last best: 1.86253)
[2026-05-29 15:52:44] [valid] Ep. 240 : Up. 15000 : perplexity 

In [5]:
!grep "learn" {MODEL_1_DIR}/config.yml
!grep "optimizer" {MODEL_1_DIR}/config.yml

learn-rate: 0.0003
optimizer: adam
optimizer-params:
optimizer-delay: 1
lr-decay-reset-optimizer: false


For now, the same modification applied to the sequence-to-sequence architecture has also been applied to the transformer architecture, namely more frequent checkpointing.

## Model 2

## Configurations

The following flags are updated in the relevant Bash file:
- `--valid-freq 5000 --save-freq 5000 --disp-freq 500` $\rightarrow$ `--valid-freq 1000 --save-freq 1000 --disp-freq 500`
- `--early-stopping 10` $\rightarrow$ `--early-stopping 5`

In [6]:
!find {ROOT}/*.sh

/home/lawun330/Desktop/basic-neural-machine-translation/seq2seq.myph.sh
/home/lawun330/Desktop/basic-neural-machine-translation/transformer.myph.sh


In [7]:
!cat {ROOT}/transformer.myph.sh

#!/bin/bash

## Written by Ye Kyaw Thu, Affiliated Professor, CADT, Cambodia
## for NMT Experiments between Burmese and Ethnic Languages
## used Marian NMT Framework for transformer training
## Last updated: 23 May 2022

model_folder="../models/change1.transformer.myph"; # -- MODIFIED --
mkdir ${model_folder};
data_path="../data/cleaned"; # -- MODIFIED --
vocab_path="../data/vocab"; # -- MODIFIED --
src="my"; tgt="ph"; # -- MODIFIED --

marian \
  --type transformer \
  --train-sets ${data_path}/train.${src} ${data_path}/train.${tgt} \
  --max-length 200 \
  --valid-sets ${data_path}/dev.${src} ${data_path}/dev.${tgt} \
  --vocabs ${vocab_path}/vocab.${src}.yml ${vocab_path}/vocab.${tgt}.yml \
  --maxi-batch 100 \
  --valid-translation-output ${model_folder}/valid.output --quiet-translation \
  --beam-size 6 --normalize 0.6 \
  --transformer-heads 8 \
  --transformer-postprocess-emb d \
  --transformer-postprocess dan \
  --transformer-dropout 0.3 --label-smoothing 0.1 \
  --learn-rate

## Machine Translation: Transformer Training for Grapheme to Phoneme

In [8]:
# check available GPU(s)
!nvidia-smi

Mon Jun  1 07:13:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.03             Driver Version: 580.159.03     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1060        Off |   00000000:01:00.0 Off |                  N/A |
| N/A   51C    P8             10W /   35W |       4MiB /   6144MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [9]:
!{ROOT}/transformer.myph.sh

[2026-06-01 07:13:02] [marian] Marian v1.12.0 65bf82ff 2023-02-21 09:56:29 -0800
[2026-06-01 07:13:02] [marian] Running on ROG-Strix-Hero-II as process 159163 with command line:
[2026-06-01 07:13:02] [marian] marian -c ../models/change1.transformer.myph/config.yml
[2026-06-01 07:13:02] [config] after: 0e
[2026-06-01 07:13:02] [config] after-batches: 0
[2026-06-01 07:13:02] [config] after-epochs: 0
[2026-06-01 07:13:02] [config] all-caps-every: 0
[2026-06-01 07:13:02] [config] allow-unk: false
[2026-06-01 07:13:02] [config] authors: false
[2026-06-01 07:13:02] [config] beam-size: 6
[2026-06-01 07:13:02] [config] bert-class-symbol: "[CLS]"
[2026-06-01 07:13:02] [config] bert-mask-symbol: "[MASK]"
[2026-06-01 07:13:02] [config] bert-masking-fraction: 0.15
[2026-06-01 07:13:02] [config] bert-sep-symbol: "[SEP]"
[2026-06-01 07:13:02] [config] bert-train-type-embeddings: true
[2026-06-01 07:13:02] [config] bert-type-vocab-size: 2
[2026-06-01 07:13:02] [config] build-info: ""
[2026-06-01 07:1

## Results: Config File

In [10]:
!cat {MODEL_2_DIR}/config.yml

# Marian configuration file generated at 2026-06-01 07:13:01 +0700 with version v1.12.0 65bf82ff 2023-02-21 09:56:29 -0800
# General options
authors: false
cite: false
build-info: ""
workspace: 1000
log: ../models/change1.transformer.myph/train.log
log-level: info
log-time-zone: ""
quiet: false
quiet-translation: true
seed: 1111
check-nan: false
interpolate-env-vars: false
relative-paths: false
sigterm: save-and-exit
# Model options
model: ../models/change1.transformer.myph/model.npz
pretrained-model: ""
ignore-model-config: false
type: transformer
dim-vocabs:
  - 0
  - 0
dim-emb: 512
factors-dim-emb: 0
factors-combine: sum
lemma-dependency: ""
lemma-dim-emb: 0
dim-rnn: 1024
enc-type: bidirectional
enc-cell: gru
enc-cell-depth: 1
enc-depth: 2
dec-cell: gru
dec-cell-base-depth: 2
dec-cell-high-depth: 1
dec-depth: 2
skip: false
layer-normalization: false
right-left: false
input-types:
  []
best-deep: false
tied-embeddings: true
tied-embeddings-src: false
tied-embeddings-all: false
output

## Results: Output Models

In [11]:
# check available models
!ls {MODEL_2_DIR}/ --color=auto

config.yml          model.iter6000.npz       model.npz.progress.yml
model.iter1000.npz  model.iter7000.npz       model.npz.yml
model.iter2000.npz  model.iter8000.npz       train.log
model.iter3000.npz  model.npz                transformer.my-ph.log
model.iter4000.npz  model.npz.decoder.yml    valid.log
model.iter5000.npz  model.npz.optimizer.npz  valid.output


In [12]:
# check best model
!cat {MODEL_2_DIR}/valid.log

[2026-06-01 07:17:24] [valid] Ep. 16 : Up. 1000 : cross-entropy : 1.97125 : new best
[2026-06-01 07:17:25] [valid] Ep. 16 : Up. 1000 : perplexity : 1.66998 : new best
[2026-06-01 07:17:25] [valid] First sentence's tokens as scored:
[2026-06-01 07:17:25] [valid] DefaultVocab keeps original segments for scoring
[2026-06-01 07:17:25] [valid]   Hyp: ou' ta- ra- hpa- la. ni
[2026-06-01 07:17:25] [valid]   Ref: ou' ta- ra. hpa- la. gu. ni
[2026-06-01 07:17:27] [valid] Ep. 16 : Up. 1000 : bleu : 64.1206 : new best
[2026-06-01 07:21:19] [valid] Ep. 32 : Up. 2000 : cross-entropy : 1.68929 : new best
[2026-06-01 07:21:20] [valid] Ep. 32 : Up. 2000 : perplexity : 1.55187 : new best
[2026-06-01 07:21:23] [valid] Ep. 32 : Up. 2000 : bleu : 69.9408 : new best
[2026-06-01 07:26:04] [valid] Ep. 48 : Up. 3000 : cross-entropy : 1.68818 : new best
[2026-06-01 07:26:04] [valid] Ep. 48 : Up. 3000 : perplexity : 1.55142 : new best
[2026-06-01 07:26:06] [valid] Ep. 48 : Up. 3000 : bleu : 71.024 : new best
[2

## Machine Translation

From the results above, the best model occurs around 5,000 iterations, achieving the best BLEU score on the validation set.

In [13]:
# check available GPU(s)
!nvidia-smi

Mon Jun  1 14:51:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.03             Driver Version: 580.159.03     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1060        Off |   00000000:01:00.0 Off |                  N/A |
| N/A   50C    P8             10W /   35W |       4MiB /   6144MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [14]:
# use the best model on test data to generate the hypothesis file
!time marian-decoder -m {MODEL_2_DIR}/model.iter5000.npz -v {VOCAB_DIR}/vocab.my.yml {VOCAB_DIR}/vocab.ph.yml --devices 0 < {DATA_DIR}/test.my > {DATA_DIR}/transformer.change1.myph.hyp.txt

[2026-06-01 14:51:50] [marian] Marian v1.12.0 65bf82ff 2023-02-21 09:56:29 -0800
[2026-06-01 14:51:50] [marian] Running on ROG-Strix-Hero-II as process 176713 with command line:
[2026-06-01 14:51:50] [marian] marian-decoder -m /home/lawun330/Desktop/basic-neural-machine-translation/models/change1.transformer.myph/model.iter5000.npz -v /home/lawun330/Desktop/basic-neural-machine-translation/data/vocab/vocab.my.yml /home/lawun330/Desktop/basic-neural-machine-translation/data/vocab/vocab.ph.yml --devices 0
[2026-06-01 14:51:50] [config] alignment: ""
[2026-06-01 14:51:50] [config] allow-special: false
[2026-06-01 14:51:50] [config] allow-unk: false
[2026-06-01 14:51:50] [config] authors: false
[2026-06-01 14:51:50] [config] beam-size: 12
[2026-06-01 14:51:50] [config] bert-class-symbol: "[CLS]"
[2026-06-01 14:51:50] [config] bert-mask-symbol: "[MASK]"
[2026-06-01 14:51:50] [config] bert-masking-fraction: 0.15
[2026-06-01 14:51:50] [config] bert-sep-symbol: "[SEP]"
[2026-06-01 14:51:50] [c

## Model Evaluation
### Multi-BLEU Score

In [15]:
# check directory where score evaluation script lives 
!ls {BLEU_DIR} --color=auto

binarize4moses2.perl    generic-parallel.perl  multi-bleu.perl
bsbleu.py               giza-parallel.perl     multi_moses.py
compound-splitter.perl  lopar2pos.pl           ph_numbers.perl
extract-factors.pl      moses-parallel.pl      qsub-wrapper.pl
extract-parallel.perl   moses_sim_pe.py        reverse-alignment.perl
fsa2fsal.pl             mteval-v11b.pl         score-parallel.perl
fsa2plf.pl              mteval-v12.pl          score_parallel.py
fsal2fsa.pl             mteval-v13a.pl         strip-xml.perl
fsa-sample.fsa          mteval-v14.pl          trainlm-irst2.perl


In [16]:
# compute BLEU score with the reference file
!perl {BLEU_DIR}/multi-bleu.perl {DATA_DIR}/test.ph < {DATA_DIR}/transformer.change1.myph.hyp.txt | tee {RESULT_DIR}/myph_transformer_change1_result.txt

BLEU = 71.49, 87.2/75.7/66.5/60.0 (BP=0.998, ratio=0.998, hyp_len=8034, ref_len=8048)
